# What House members traded in 2026, and how late they reported it

Every stock trade House members reported for 2026: the split by party and type, how many days each report came after the trade, who filed past 45 days, and the stocks most members bought.

Read it with charts at https://www.quantisnow.com/sec-api/notebooks/congress-stock-trades-2026. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

Members of the House report their trades on periodic transaction reports. The STOCK Act gives them 45 days from the trade. This notebook pulls every stock trade dated in 2026 and checks what they bought, what they sold and how long they took to say so.

The helper pages through a query 50 records at a time, one API call per page. Every notebook here starts with it.

In [1]:
import os, requests
import pandas as pd

API = "https://www.quantisnow.com/api/v1"
KEY = os.environ["QUANTISNOW_API_KEY"]

def search(dataset, query, start=0, size=50, sort=None):
    body = {"query": query, "from": start, "size": size}
    if sort:
        body["sort"] = sort
    r = requests.post(f"{API}/{dataset}", json=body,
                      headers={"Authorization": f"Bearer {KEY}"})
    r.raise_for_status()
    return r.json()

def search_all(dataset, query, sort=None):
    """Every record that matches, 50 per call."""
    rows, start = [], 0
    while True:
        page = search(dataset, query, start, 50, sort)
        rows += page["data"]
        start += 50
        if start >= min(page["total"]["value"], 10_000):
            return rows

## Pull the trades

`assetType:ST` keeps stocks and leaves out bonds, funds and options.

In [2]:
trades = pd.json_normalize(search_all("congress/trades",
    "assetType:ST AND tradedOn:[2026-01-01 TO 2026-11-21]"))
print(f"{len(trades):,} stock trades by {trades['member.name'].nunique()} members")

1,382 stock trades by 66 members

In [3]:
trades["type"].value_counts().rename("trades")

type,trades
purchase,538
sale,506
partial_sale,314
exchange,24


In [4]:
trades.groupby("member.party").agg(trades=("type", "size"), members=("member.name", "nunique"))

member.party,trades,members
D,495,29
R,887,37


Republicans reported 887 of the 1,382 trades and Democrats 495. A few members do most of the trading, so per-member counts swing these totals more than party does.

## How long after the trade

In [5]:
trades["lag"] = (pd.to_datetime(trades["filedOn"]) - pd.to_datetime(trades["tradedOn"])).dt.days
bins = pd.cut(trades["lag"], [-1, 15, 30, 45, 90, 10_000],
              labels=["0-15", "16-30", "31-45", "46-90", "over 90"])
bins.value_counts(sort=False).plot.bar(title="Days from trade to report, 2026 stock trades")

Days from trade to report, 2026 stock trades
0-15	519
16-30	560
31-45	286
46-90	9
over 90	8

In [6]:
late = trades[trades["lag"] > 45]
print(f"median: {int(trades['lag'].median())} days")
print(f"reported after 45 days: {len(late)} trades by {late['member.name'].nunique()} members")

median: 21 days
reported after 45 days: 17 trades by 6 members

In [7]:
(late.groupby("member.name")
     .agg(late_trades=("lag", "size"), longest_days=("lag", "max"))
     .sort_values(["late_trades", "longest_days"], ascending=False).head(10))

member.name,late_trades,longest_days
Christian D. Menefee,5,111
Rick W. Allen,5,51
Laurel M. Lee,4,69
Shri Thanedar,1,216
"Gilbert Ray Cisneros, Jr.",1,158
Tim Walberg,1,113


Most reports arrive within a month: the median is 21 days, and 1,365 of the 1,382 trades were reported inside 45 days. Seventeen were not. Christian Menefee and Rick Allen had five late trades each, and Shri Thanedar's one late trade was reported 216 days after it was made.

## What they bought

Count members, not trades. One member buying the same stock in twelve accounts would otherwise top the list.

In [8]:
buys = trades[(trades["type"] == "purchase") & trades["ticker"].notna()]
popular = buys.groupby("ticker")["member.name"].nunique().sort_values(ascending=False).head(12)
popular.plot.barh(title="Stocks the most House members bought, 2026")

Stocks the most House members bought, 2026
MSFT	8
META	5
AAPL	4
AMZN	4
HD	4
NVDA	4
SPCX	4
ABT	3
AMAT	3
AMD	3
BRK.B	3
GOOGL	3

Microsoft drew the most buyers, eight members. SPCX, SpaceX's ticker since its listing, already has four.

## The largest reported trades

Reports give a dollar range, not an amount. Sort by the low end so nothing is overstated.

In [9]:
big = trades[trades["amount.min"] >= 500_001].sort_values(["amount.min", "tradedOn"], ascending=[False, True])
big[["member.name", "ticker", "type", "tradedOn", "amount.min", "amount.max"]].head(12)

member.name,ticker,type,tradedOn,amount.min,amount.max
Nancy Pelosi,BE,purchase,2026-07-24,"1,000,001","5,000,000"
Nancy Pelosi,AMZN,purchase,2026-01-16,"500,001","1,000,000"
Nancy Pelosi,GOOGL,purchase,2026-01-16,"500,001","1,000,000"
Kevin Hern,TXN,sale,2026-03-18,"500,001","1,000,000"
Kevin Hern,SMA,sale,2026-03-18,"500,001","1,000,000"
Sara Jacobs,QCOM,partial_sale,2026-05-06,"500,001","1,000,000"
Sara Jacobs,QCOM,partial_sale,2026-05-07,"500,001","1,000,000"
Nancy Pelosi,INTC,purchase,2026-07-24,"500,001","1,000,000"
Nancy Pelosi,BE,purchase,2026-07-28,"500,001","1,000,000"
Kevin Hern,OKE,exchange,2026-09-11,"500,001","1,000,000"


Nancy Pelosi's account reported five of the ten trades of $500,001 or more, all purchases, including Bloom Energy at $1 million to $5 million in July. Kevin Hern and Sara Jacobs made the largest sales.

## Take it further

Filter one member with `member.name:"Nancy Pelosi"` or one stock with `ticker:NVDA`. `owner:spouse` separates trades in a spouse's account. The [Congress trades dataset page](https://www.quantisnow.com/sec-api/congress-trades) lists every field.